# Residuals

In [ ]:
from datascience import *
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

## Minimizing RMSE: Nonlinear Regression

In [ ]:
shotput = Table.read_table('shotput.csv')
shotput

In [ ]:
shotput.scatter('Weight Lifted')

### Linear Regression

$\text{estimate of Shot Put Distance} = a \cdot \text{Weight Lifted} + b$

In [ ]:
def shotput_linear_rmse(any_slope, any_intercept):
    x = shotput.column('Weight Lifted')
    y = shotput.column('Shot Put Distance')
    estimate = any_slope*x + any_intercept
    return np.mean((y - estimate) ** 2) ** 0.5

In [ ]:
best_line = minimize(shotput_linear_rmse)
best_line

In [ ]:
weights = shotput.column(0)

In [ ]:
linear_fit = best_line.item(0)*weights + best_line.item(1)

shotput.with_column(
    'Best Line', linear_fit
).scatter(0)

### Quadratic Regression

**Quadratic Function**

$$
f(x) ~=~ ax^2 + bx + c
$$
for constants $a$, $b$, and $c$.

$\begin{align}
\text{estimate of Shot Put Distance} =& a \cdot (\text{Weight Lifted})^2 \\
&+ b \cdot \text{Weight Lifted} + c
\end{align}$

Judge, Lawrence W.; Bellar, David; Thrasher, Ashley; Simon, Laura; Hindawi, Omar S.; and Wanless, Elizabeth (2013) "A Pilot Study Exploring the Quadratic Nature of the Relationship of Strength to Performance Among Shot Putters," International Journal of Exercise Science: Vol. 6 : Iss. 2, Pages 171 - 179. https://doi.org/10.70252/XNTZ5949


In [ ]:
def shotput_quadratic_rmse(a, b, c):
    x = shotput.column('Weight Lifted')
    y = shotput.column('Shot Put Distance')
    estimate = a*(x**2) + b*x + c
    return np.mean((y - estimate) ** 2) ** 0.5

In [ ]:
best_quad = minimize(shotput_quadratic_rmse)
best_quad

In [ ]:
# x = weight lifted = 100 kg
# Then predicted shot put distance (m):

(-0.00104)*(100**2) + 0.2827*100 - 1.5318

In [ ]:
weights = shotput.column(0)
quad_fit = best_quad.item(0)*(weights**2) + best_quad.item(1)*weights + best_quad.item(2)

In [ ]:
shotput.with_column('Best Quadratic Curve', quad_fit).scatter(0)

### Comparing the two regressions

In [ ]:
best_line

In [ ]:
shotput_linear_rmse(best_line.item(0), best_line.item(1))

In [ ]:
best_quad

In [ ]:
shotput_quadratic_rmse(best_quad.item(0), best_quad.item(1), best_quad.item(2))

## Regression-Related Functions

In [ ]:
def standard_units(arr):
    """ Converts an array to standard units """
    return (arr - np.average(arr))/np.std(arr)

def correlation(t, x, y):
    """ Computes correlation: t is a table, and x and y are column names """
    x_standard = standard_units(t.column(x))
    y_standard = standard_units(t.column(y))
    return np.average(x_standard * y_standard)

def slope(t, x, y):
    """ Computes the slope of the regression line, like correlation above """
    r = correlation(t, x, y)
    y_sd = np.std(t.column(y))
    x_sd = np.std(t.column(x))
    return r * y_sd / x_sd

def intercept(t, x, y):
    """ Computes the intercept of the regression line, like slope above """
    x_mean = np.mean(t.column(x))
    y_mean = np.mean(t.column(y))
    return y_mean - slope(t, x, y)*x_mean

def fitted_values(t, x, y):
    """Return an array of the regression estimates (predictions) at all the x values"""
    a = slope(t, x, y)
    b = intercept(t, x, y)
    return a*t.column(x) + b

## Plotting Residuals

### Case study 1: 2016 election demographics

In [ ]:
demographics = Table.read_table('district_demographics2024.csv')
demographics.sample(6).show()

**Linear Fit**

$\text{estimate of Median Income} = a \cdot \text{College\%} + b$

In [ ]:
# fit line, then plot fitted
fitted_income = demographics.select('College%', 'Median Income')
fitted_income = fitted_income.with_columns('Fitted Median Income',
    fitted_values(demographics, 'College%', 'Median Income'))
fitted_income.scatter('College%')
fitted_income.sample(5).show()

In [ ]:
def residuals(t, x, y):
    predictions = fitted_values(t, x, y)
    return t.column(y) - predictions

In [ ]:
income_residuals = fitted_income.with_columns(
    'Residual', residuals(demographics, 'College%', 'Median Income')
)
income_residuals.show(5)

These two functions will help us plot things the rest of the way.

In [ ]:
def plot_fitted(t, x, y):
    fitted_label = 'Fitted ' + y
    (t.with_column(fitted_label, fitted_values(t, x, y))
      .select(x, y, fitted_label).scatter(0))

def plot_residuals(t, x, y):
    residuals_label = y + ' Residuals'
    (t.with_columns(residuals_label, residuals(t, x, y))
      .scatter(x, residuals_label))

______

In [ ]:
plot_fitted(demographics, 'College%', 'Median Income')

In [ ]:
plot_residuals(demographics, 'College%', 'Median Income')

**Residual plots can tell us about our model fit.**

### Case study 2: dugongs 

In [ ]:
dugong = Table.read_table('dugong.csv')
dugong.show(5)

In [ ]:
dugong.scatter('Length', 'Age')

In [ ]:
correlation(dugong, 'Length', 'Age')

$\text{estimate of Age} = a \cdot \text{Length} + b$

In [ ]:
plot_fitted(dugong, 'Length', 'Age')

In [ ]:
plot_residuals(dugong, 'Length', 'Age')

### Case study 3: US women

In [ ]:
us_women = Table.read_table('us_women.csv')
us_women.show(5)

In [ ]:
us_women.scatter('height')

In [ ]:
correlation(us_women, 'height', 'avg weight')

$\text{estimate of height} = a \cdot \text{average weight} + b$

In [ ]:
plot_fitted(us_women, 'height', 'avg weight')

In [ ]:
plot_residuals(us_women, 'height', 'avg weight')

### Case study 4: Heights

In [ ]:
family_heights = Table.read_table('family_heights.csv')
family_heights.where('family', '1')

In [ ]:
parents = (family_heights.column('father') + family_heights.column('mother'))/2
heights = Table().with_columns(
    'Parent Average', parents,
    'Child', family_heights.column('child')
    )
heights.show(6)

$\text{estimate of a child's adult height} = a \cdot \text{average parent height} + b$

In [ ]:
plot_fitted(heights, 'Parent Average', 'Child')

In [ ]:
correlation(heights, 'Parent Average', 'Child')

In [ ]:
plot_residuals(heights, 'Parent Average', 'Child')

## Mathematical Properties of Residuals

 The average of the residuals are zero.

In [ ]:
round(np.average(residuals(demographics, 'College%', 'Median Income')), 10)

In [ ]:
round(np.average(residuals(dugong, 'Length', 'Age')), 10)

In [ ]:
round(np.average(residuals(us_women, 'height', 'avg weight')), 10)

______

The correlation coefficients between:

- The residuals and $x$
- The residuals and the fitted values

are both 0.

In [ ]:
dugong = dugong.with_columns(
    'Residual', residuals(dugong, 'Length', 'Age'),
    'Fitted Age', fitted_values(dugong, 'Length', 'Age')
)
dugong.show(3)

In [ ]:
dugong.drop('Residual').scatter('Length')

In [ ]:
dugong.scatter('Length', 'Residual')

In [ ]:
round(correlation(dugong, 'Length', 'Residual'), 10)

In [ ]:
round(correlation(dugong, 'Fitted Age', 'Residual'), 10)

In [ ]:
round(correlation(dugong, 'Age', 'Residual'), 10)

### Residual SD

In [ ]:
dugong.sample(3)

In [ ]:
dugong.hist('Residual')

In [ ]:
np.std(dugong.column('Residual'))

In [ ]:
r = correlation(dugong, 'Age', 'Length')
r

### $r$ and the SD of Fitted Values

In [ ]:
dugong.scatter('Length', ['Age', 'Fitted Age'])

In [ ]:
dugong.hist('Fitted Age', bins=np.arange(0, 32, 4))

In [ ]:
dugong.hist('Age', bins=np.arange(0, 32, 4))